In [ ]:
# imports
import json
import requests
import time
import numpy as np
import re
import mwparserfromhell
from collections import defaultdict
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv
from pathlib import Path
from langchain_core.documents import Document
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer


# Cleaning + Chunking + Embedding + Upload to Pinecone

### Movie data prep 
- chunking (384 words + 75 overlap)
- organizing metadata with index m_{tmdb_id}_{chunk_index}


The reason to set chunk size to 384 is that the embedding model of choice sentence-transformers/all-mpnet-base-v2 work the best with 384 words.

In [ ]:
# Load movie data (example - adjust path as needed)
try:
    with open("../raw_data/movie_raw/tmdb_wikipedia_plots_cleaned.json", "r", encoding="utf-8") as f:
        movie_data = json.load(f)
    print(f"✓ Loaded {len(movie_data)} movies")
except FileNotFoundError:
    print("json not found. Please load your movie data first.")
    movie_data = []

# make sure chunk_data folder    
chunk_dir = Path("../chunk_data")

# Create folder only if it doesn't exist
if not chunk_dir.exists():
    chunk_dir.mkdir(parents=True)
    print("Created folder:", chunk_dir)
else:
    print("Folder already exists, skipping:", chunk_dir)


✓ Loaded 613 movies


In [ ]:
# langchain recursive text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1700,
    chunk_overlap=450
)

all_chunks = []
id = 0
used_ids = False

for movie in movie_data:
    movie_plot = (
        movie["plot"].replace("\n", " ")
        .replace("Plot", "")
        .replace("plot", "")
        .replace("PLOT", "")
        .strip()
    )
    chunks = text_splitter.split_text(movie_plot)
    for j, chunk in enumerate(chunks):
        if movie['tmdb_id'] is not None:
            all_chunks.append({
                "id": f"m_{movie['tmdb_id']}_{j}",
                "text": chunk,
                "metadata": {
                    "title": movie["title"],
                    "year": movie["year"],
                    "wikipedia_id": movie["wikipedia_title"],
                }
            })
        else:
            all_chunks.append({
                "id": f"m_{id}_{j}",
                "text": chunk,
                "metadata": {
                    "title": movie["title"],
                    "year": movie["year"],
                    "wikipedia_id": movie["wikipedia_title"],
                }
            })
            used_ids = True
    if used_ids:
      id += 1

# Write valid JSON
with open("../chunk_data/movie_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

### TV Show data prep 
- organizing metadata with index t_{tmdb_id}_{chunk_index}


In [ ]:
# Load tv data
try:
    with open("/../raw_data/tv_raw/filtered_episode_dicts.json", "r", encoding="utf-8") as f:
        show_data = json.load(f)
    print(f"✓ Loaded {len(show_data)} movies")
except FileNotFoundError:
    print("json not found. Please load your movie data first.")
    show_data = []

✓ Loaded 133 movies


In [ ]:
all_chunks = []
index = 0
title = show_data[0]["title"]
previous_title = show_data[0]["title"]
for show in show_data:
    if show['tmdb_id'] is not None:
        all_chunks.append({
            "id": f"t_{show['tmdb_id']}_{show['episode_number'] - 1}",
            "text": show[f"{show['title']}_episode{show['episode_number']}"],
            "metadata": {
                "title": show["title"],
                "year": show["year"],
                "episode_number": show["episode_number"]
        }})
    else:
        all_chunks.append({
            "id": f"t_{index}_{show['episode_number'] - 1}",
            "text": show[f"{show['title']}_episode{show['episode_number']}"],
            "metadata": {
                "title": show["title"],
                "year": show["year"],
                "episode_number": show["episode_number"]
        }})
        title = show['title']
        if title != previous_title:
            index += 1
        previous_title = title
    
# Write valid JSON
with open("../chunk_data/tv_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

### Post-processing
- remove wrong data

In [ ]:
try:
    with open("../chunk_data/tv_chunks.json", "r", encoding="utf-8") as f:
        tv_data = json.load(f)
    print(f"✓ Loaded {len(tv_data)} TV chunks")
except FileNotFoundError:
    print("json not found. Please load your TV data first.")
    tv_data= []

✓ Loaded 11252 TV chunks


In [ ]:
# Wikipedia API Functions for filtering TV chunks
WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": "WikipediaTVShowExtractor/1.0 (https://example.com/contact; your-email@example.com)"
}

def search_wikipedia(title, year=None, media_type=None):
    """Search for a Wikipedia page by title"""
    query = title
    if media_type == "tv":
        query += " TV series"
    elif media_type == "movie":
        query += " film"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    }

    try:
        r = requests.get(WIKI_API, params=params, headers=HEADERS)
        r.raise_for_status()
        results = r.json()["query"]["search"]
        time.sleep(0.5)  # Be respectful to Wikipedia's servers
        return results[0]["title"] if results else None
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            print(f"⚠️  403 Forbidden - Wikipedia may be blocking requests. Try again later.")
        raise


In [ ]:
# Create mapping of {db_title: wikipedia_title} for TV chunks to filter out wrong matches
print("Creating title mapping...")
print("=" * 80)

# Get unique titles from TV chunks
unique_titles = set()
for chunk in tv_data:
    title = chunk["metadata"]["title"]
    unique_titles.add(title)

print(f"Unique TV show titles: {len(unique_titles)}")

# Create mapping dictionary
title_mapping = {}

# Process each title with progress bar
for title in tqdm(unique_titles, desc="Mapping titles"):
    try:
        # Search Wikipedia
        wiki_title = search_wikipedia(title, media_type="tv")
        
        if wiki_title:
            title_mapping[title] = wiki_title
        else:
            # No Wikipedia page found - map to None
            title_mapping[title] = None
    
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            print(f"\n⚠️  403 Forbidden detected! Waiting 60 seconds...")
            time.sleep(60)
            # Retry this title
            try:
                wiki_title = search_wikipedia(title, media_type="tv")
                title_mapping[title] = wiki_title if wiki_title else None
            except:
                title_mapping[title] = None
        else:
            print(f"\n⚠️  Error for '{title}': {e}")
            title_mapping[title] = None
    
    except Exception as e:
        print(f"\n⚠️  Unexpected error for '{title}': {e}")
        title_mapping[title] = None

print("\n" + "=" * 80)
print("Mapping complete!")
print(f"  Total titles mapped: {len(title_mapping)}")
print(f"  Titles with Wikipedia match: {sum(1 for v in title_mapping.values() if v is not None)}")
print(f"  Titles without Wikipedia match: {sum(1 for v in title_mapping.values() if v is None)}")
print("=" * 80)

# Save mapping to file
with open("../chunk_data/tv_title_mapping.json", "w", encoding="utf-8") as f:
    json.dump(title_mapping, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved title mapping to ../chunk_data/tv_title_mapping.json")
print(f"  Mapping format: {{'TMDB Title': 'Wikipedia Title'}}")

Creating title mapping...
Unique TV show titles: 744


Mapping titles: 100%|██████████| 744/744 [11:55<00:00,  1.04it/s]


Mapping complete!
  Total titles mapped: 744
  Titles with Wikipedia match: 744
  Titles without Wikipedia match: 0

✓ Saved title mapping to ../chunk_data/tv_title_mapping.json
  Mapping format: {'TMDB Title': 'Wikipedia Title'}


In [ ]:
# Manual inspection for the matches and remove unmatched title (can also be done with code but are less accurate)
# keep only shows remain in the mapping list (those that are correctly matched)
print("Filtering TV chunks based on title mapping...")
print("=" * 80)

# Load the mapping if it exists (in case we're running this cell separately)
if 'title_mapping' not in locals():
    try:
        with open("../chunk_data/tv_title_mapping.json", "r", encoding="utf-8") as f:
            title_mapping = json.load(f)
        print(f"✓ Loaded title mapping with {len(title_mapping)} titles")
    except FileNotFoundError:
        print("⚠️  Title mapping file not found. Please run the mapping cell first.")
        title_mapping = {}

# Get set of valid titles (keys in the mapping)
valid_titles = [set(title_mapping.keys())]

print(f"Valid titles in mapping: {len(valid_titles)}")

# Filter tv_data to only keep chunks with titles in the mapping
filtered_tv_data = []
for chunk in tv_data:
    title = chunk["metadata"]["title"]
    if title in valid_titles:
        filtered_tv_data.append(chunk)

print(f"\nFiltering results:")
print(f"  Original chunks: {len(tv_data)}")
print(f"  Filtered chunks: {len(filtered_tv_data)}")
print(f"  Removed chunks: {len(tv_data) - len(filtered_tv_data)}")
print("=" * 80)

# Update tv_data with filtered data
tv_data = filtered_tv_data
print(f"\n✓ Updated tv_data with {len(tv_data)} filtered chunks")

Filtering TV chunks based on title mapping...
Valid titles in mapping: 472

Filtering results:
  Original chunks: 11252
  Filtered chunks: 5853
  Removed chunks: 5399

✓ Updated tv_data with 5853 filtered chunks


In [ ]:
# Save filtered TV chunks back to file
with open("../chunk_data/tv_chunks.json", "w", encoding="utf-8") as f:
    json.dump(tv_data, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(tv_data)} filtered TV chunks to ../chunk_data/tv_chunks.json")

✓ Saved 5853 filtered TV chunks to ../chunk_data/tv_chunks.json


### Merge show and movie data into one single file

In [ ]:
# load chunks
movie_chunks = json.load(open("../chunk_data/movie_chunks.json", "r", encoding="utf-8"))
show_chunks = json.load(open("../chunk_data/tv_chunks.json", "r", encoding="utf-8"))

# Merge chunks
merged_chunks = []

# Add movie chunks
merged_chunks.extend(movie_chunks)

# Add TV show chunks
merged_chunks.extend(show_chunks)

# Optional: write to a single JSON file
with open("../chunk_data/merged_chunks.json", "w", encoding="utf-8") as f:
    json.dump(merged_chunks, f, indent=2, ensure_ascii=False)

print(f"Total chunks: {len(merged_chunks)}")


Total chunks: 6761


### Create embedding 

Why sentence-transformers/all-mpnet-base-v2  for embedding?

Among all embedding models, 
I picked sentence-transformers/all-mpnet-base-v2 because it produces embeddings suitable for text chunks of around 384 tokens projected to a 768 dimensional dense vector space, which aligns well with the typical length of movie or show plot segments. This makes it ideal for capturing the semantic meaning of individual plot events without splitting important context, ensuring effective retrieval and relevance when querying the plot database.


In [40]:
import json
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# Load merged chunks
with open("../chunk_data/merged_chunks.json", "r", encoding="utf-8") as f:
    merged_chunks = json.load(f)

print(f"Loaded {len(merged_chunks)} chunks")

# Extract text
texts = [chunk.get("text", "") for chunk in merged_chunks]

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Embed in batches
BATCH_SIZE = 16
all_embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding batches"):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_embeddings = model.encode(
        batch_texts,
        show_progress_bar=False,
        convert_to_numpy=True,
        device="cpu"
    )
    # Convert numpy arrays to lists
    all_embeddings.extend([emb.tolist() for emb in batch_embeddings])

# Add embeddings to chunks
for chunk, embedding in zip(merged_chunks, all_embeddings):
    chunk["embedding"] = embedding

# Save embedded chunks
OUTPUT_FILE = "../chunk_data/merged_chunks_with_embeddings.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(merged_chunks, f, indent=2, ensure_ascii=False)

print(f"Saved {len(merged_chunks)} chunks with embeddings to {OUTPUT_FILE}")
print(f"Embedding dimension: {len(all_embeddings[0])}")

Loaded 7951 chunks


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1520.63it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

### Embedding metadata cleaning

In [38]:
cleaned = []
with open("../chunk_data/merged_chunks_with_embeddings.json", "r", encoding="utf-8") as f:
    embedded_chunks = json.load(f)

for chunk in embedded_chunks:
    if chunk["id"][0] == "m":
        cleaned_chunks = {
            "id": chunk["id"],
            "values": chunk["embedding"],
            "metadata": {
                "id": chunk["id"],
                "title": chunk["metadata"]["title"],
                "year": chunk["metadata"]["year"],
                "type": "movie"
            }
        }
    else:
        cleaned_chunks = {
            "id": chunk["id"],
            "values": chunk["embedding"],
            "metadata": {
                "id": chunk["id"],
                "title": chunk["metadata"]["title"],
                "year": chunk["metadata"]["year"],
                "episode_number": chunk["metadata"]["episode_number"],
                "type": "show"
            }
        }
    cleaned.append(cleaned_chunks)

In [39]:
# write to file
with open("../chunk_data/merged_chunks_with_embeddings_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

### Upload to pinecone (or any other vector db you are using)

In [41]:
# read in embeddings
with open("../chunk_data/merged_chunks_with_embeddings_cleaned.json", "r", encoding="utf-8") as f:
    vectors = json.load(f)

# define test size
TEST_SIZE = 50

# split into test and upload
test_vectors = vectors[:TEST_SIZE]      # keep locally
upload_vectors = vectors[TEST_SIZE:]    # send to Pinecone

load_dotenv()

# Get pinecone API key from environment variable
pc_key = os.getenv("PINECONE_API_KEY")
if not pc_key:
    raise ValueError("PINECONE_API_KEY not found in environment variables. Please create a .env file with your API key.")


pc = Pinecone(api_key=pc_key)

INDEX_NAME = "stream-embeddings"
DIMENSION = len(upload_vectors[0]["values"])  # auto-detect
METRIC = "cosine"

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# create index object
index = pc.Index(INDEX_NAME)

In [42]:
# upload in batches
BATCH_SIZE = 100  # adjust as needed

for i in tqdm(range(0, len(upload_vectors), BATCH_SIZE), desc="Uploading vectors"):
    batch = upload_vectors[i:i+BATCH_SIZE]
    index.upsert(vectors=batch)

Uploading vectors: 100%|██████████| 80/80 [00:58<00:00,  1.37it/s]
